# 2주차 · 2교시 통계적 추론 + 회귀 과제

**과제 유형: A + B**

- 유형 A (기초 체화형): CLT 시뮬레이션과 신뢰구간을 직접 계산·해석합니다. GPT에 코드를 통째로 시키기보다, 손으로 값을 만들어 보며 개념을 몸에 익히는 부분입니다.
- 유형 B (도구 활용형): statsmodels/sklearn 같은 도구를 씁니다. GPT 사용은 허용하지만 **결과가 맞는지 판단·검증·해석하는 것은 반드시 본인이** 합니다.

## 과제 목적

1교시에서 우리는 `tips` 데이터의 분포·상관·이상치·결측을 눈으로 살펴봤습니다(EDA/기술통계). 이번 과제의 목적은 그 "눈으로 본 패턴"을 두 방향으로 확장하는 것입니다. 첫째는 **추론**입니다. 표본(우리가 가진 244행)에서 본 평균/차이가 우연인지, 아니면 모집단에서도 의미 있는지를 표본분포·신뢰구간·가설검정으로 판단합니다. 둘째는 **회귀**입니다. `total_bill`이 오르면 `tip`이 얼마나 오르는지를 수식(모델)로 정량화하고, 그 계수가 통계적으로 믿을 만한지, 모델이 데이터를 얼마나 설명하는지, 그리고 그 모델이 새 데이터를 얼마나 잘 **예측**하는지까지 이어서 봅니다. 하나의 데이터셋(`tips`)으로 로드 → EDA → 추론 → 회귀 → 해석/예측을 관통하는 프로젝트 흐름을 완성하는 것이 이 과제의 목표입니다.

각 단계의 코드에는 빈칸(`# TODO: ____`, `____`)이 많습니다. 빈칸을 채우면서 "지금 나는 어느 단계에서 무엇을 하고 있는가"를 계속 스스로 확인하세요.

## 0. 환경 설정

필요한 라이브러리를 불러오고, **본인만의 seed**를 설정합니다. seed(시드)는 난수 생성의 출발점입니다. 같은 seed를 쓰면 랜덤 결과가 항상 똑같이 재현되고, 사람마다 다른 seed를 쓰면 각자 조금씩 다른 표본을 뽑게 됩니다. 이번 과제에서는 **부트스트랩 표본추출**과 **train/test 분할**에 이 seed를 씁니다. 학번 또는 기수 뒷자리 등 본인만의 숫자를 넣으세요.

In [ ]:
# 기본 라이브러리
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# (선택) Colab에서 그래프의 한글 라벨이 네모(□)로 깨지지 않도록 한글 폰트 설정
!pip -q install koreanize-matplotlib
import koreanize_matplotlib
plt.rcParams["axes.unicode_minus"] = False

# 통계/모델링 라이브러리
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# TODO: ____ 본인만의 seed를 정수로 설정하세요 (예: 학번 뒷 4자리)
MY_SEED = ____

# 참고: 아래 np.random.seed는 numpy '전역 기본 난수'의 재현용이다.
# 이 과제의 부트스트랩·CLT는 서로 독립적으로 재현되도록 np.random.default_rng(MY_SEED)로
# 개별 생성기를 따로 만들어 쓰고, train/test 분할은 random_state=MY_SEED로 넘긴다.
np.random.seed(MY_SEED)
print('내 seed:', MY_SEED)

## 1. 필수 과제

아래 흐름을 순서대로 완성합니다. 앞 단계의 결과(변수)가 뒤 단계로 이어집니다.

데이터 로드 → EDA/상관 → 표본분포·CLT 시뮬레이션 → 신뢰구간 → 두 집단 평균차 t검정 → 단순선형회귀 → 잔차진단 → 다중회귀 → 계수유의성·R²/Adj R²·VIF → 해석 vs 예측 → 결론

### 1.1 데이터 로드

`seaborn`의 `tips` 데이터를 불러옵니다. 이 데이터는 식당 손님별 청구액(`total_bill`), 팁(`tip`), 성별(`sex`), 흡연 여부(`smoker`), 요일(`day`), 시간대(`time`), 일행 수(`size`)를 담고 있습니다.

In [ ]:
# TODO: ____ tips 데이터셋을 불러오세요
tips = sns.load_dataset("____")

# 앞부분 확인
print(tips.shape)
tips.head()

In [ ]:
# TODO: ____ 각 열의 자료형(dtype)과 결측 여부를 확인하세요
tips.____()   # info

# 결측치 개수 확인 (열별 합계)
print(tips.isnull().sum())

# TODO: ____ 수치형 변수의 기술통계(평균/표준편차/사분위수)를 확인하세요
tips.____()   # describe

### 1.2 EDA / 상관

1교시 내용을 짧게 복습합니다. 수치형 변수 간 상관계수와 산점도를 봅니다. 상관계수(correlation)는 두 변수가 함께 커지고 작아지는 정도를 -1~1로 나타낸 값입니다.

In [ ]:
# TODO: ____ 수치형 변수(total_bill, tip, size) 간 상관계수 행렬을 구하세요
num_cols = ["total_bill", "tip", "size"]
corr = tips[num_cols].____()
print(corr)

# total_bill vs tip 산점도
plt.figure(figsize=(5,4))
# TODO: ____ x=total_bill, y=tip 산점도를 그리세요
plt.scatter(tips["____"], tips["____"], alpha=0.5)
plt.xlabel("total_bill")
plt.ylabel("tip")
plt.title("total_bill vs tip")
plt.show()

**[해석 질문 1]** `total_bill`과 `tip`의 상관계수는 얼마인가요? **본인이 구한 corr 값을 소수 셋째 자리까지** 적고, 산점도의 모양과 상관계수 값이 서로 일치하는지 설명하세요. 또 산점도에서 **우상향 추세에서 벗어나는 예외 점을 하나 지목**해 보세요. (아래에 2~3문장으로)

(여기에 답)

### 1.3 표본분포 · CLT 시뮬레이션 (유형 A)

이 부분은 도구에 맡기지 말고 **직접** 만들어 보세요.

개념: 우리가 가진 244개의 `total_bill`을 잠시 "모집단"이라고 가정합니다. 여기서 크기 `n`짜리 표본을 뽑아 **평균**을 구하는 일을 아주 여러 번 반복하면, 그 평균들이 이루는 분포가 생깁니다. 이것을 **표본평균의 분포(표본분포)**라고 합니다. **중심극한정리(CLT)**는 표본 크기 `n`이 커질수록 이 표본평균의 분포가 원래 데이터 모양과 상관없이 **정규분포(종 모양)에 가까워지고, 그 퍼짐(표준편차)은 좁아진다**는 정리입니다.

여기서 개별 데이터가 흩어진 정도인 **표준편차(SD)**와, 표본평균이 표본마다 흔들리는 정도인 **표준오차(SE = σ/√n)**를 구분하는 것이 중요합니다. CLT가 말하는 "퍼짐이 좁아진다"는 바로 이 표준오차가 √n에 반비례해 줄어든다는 뜻이고, 이것이 뒤(1.4)의 신뢰구간 폭과 직접 연결됩니다.

In [ ]:
population = tips["total_bill"].values
pop_mean = population.mean()
print("모집단(가정) 평균:", round(pop_mean, 3))

rng = np.random.default_rng(MY_SEED)   # 내 seed 기반 난수 생성기

def sample_means(pop, n, n_iter=2000):
    """pop에서 크기 n 표본을 n_iter번 뽑아 각 표본평균을 반환"""
    means = []
    for _ in range(n_iter):
        # TODO: ____ 복원추출로 크기 n 표본을 뽑으세요 (rng.choice 사용)
        sample = rng.choice(pop, size=____, replace=True)
        # TODO: ____ 이 표본의 평균을 means에 추가하세요
        means.append(sample.____())
    return np.array(means)

# 표본 크기를 바꿔 가며 표본평균 분포를 비교
# 마지막 열: 이론적 표준오차 σ/√n — 경험적으로 구한 표준편차와 얼마나 맞는지 비교해 보세요
for n in [2, 10, 30, 100]:
    m = sample_means(population, n=n)
    se_theory = population.std() / np.sqrt(n)
    print(f"n={n:3d} | 표본평균들의 평균={m.mean():.3f}, 경험 표준편차={m.std():.3f}, 이론 σ/√n={se_theory:.3f}")

In [ ]:
# n=2 와 n=30 의 표본평균 분포를 히스토그램으로 비교
fig, axes = plt.subplots(1, 2, figsize=(10,4))

m_small = sample_means(population, n=2)
m_large = sample_means(population, n=30)

# TODO: ____ 왼쪽에 m_small, 오른쪽에 m_large 히스토그램을 그리세요
axes[0].hist(____, bins=30)
axes[0].set_title("표본평균 분포 (n=2)")
axes[1].hist(____, bins=30)
axes[1].set_title("표본평균 분포 (n=30)")
plt.show()

**[해석 질문 2]** `n`이 커질수록 (a) 표본평균들의 표준편차와 (b) 분포 모양은 각각 어떻게 변했나요? **본인 출력에서 n=2와 n=30의 경험 표준편차 실제 값을 인용**하고, 그 값이 이론값 σ/√n과 얼마나 잘 맞는지도 적으세요. 이것이 CLT와 어떻게 연결되나요?

(여기에 답)

### 1.4 신뢰구간 (유형 A · 부트스트랩, MY_SEED 사용)

앞의 CLT는 `total_bill`로 원리를 봤고, 이제 우리가 실제로 궁금한 값인 **평균 팁(`tip`)**에 그 원리를 적용해 신뢰구간을 만듭니다.

개념: **신뢰구간(confidence interval)**은 표본에서 계산하는 구간으로, **같은 방식으로 표본을 반복해 구간을 만들면 그 중 약 95%가 모집단의 참값(예: 평균 팁)을 포함**합니다. 이미 만들어진 하나의 구간을 두고 "참값이 이 구간 안에 있을 확률이 95%"라고 말하는 것은 **흔한 오해**입니다(참값은 고정돼 있고, 무작위로 변하는 것은 구간 쪽입니다). 여기서는 **부트스트랩**으로 만듭니다. 부트스트랩은 가진 데이터에서 **복원추출**로 같은 크기의 가짜 표본을 여러 번 만들어, 그때마다 통계량(평균)을 계산해 그 분포로 구간을 잡는 방법입니다. 95% 신뢰구간은 그 분포의 2.5% ~ 97.5% 지점입니다.

In [ ]:
tip_values = tips["tip"].values
n = len(tip_values)
boot_means = []

boot_rng = np.random.default_rng(MY_SEED)   # 내 seed 사용 -> 각자 결과가 조금씩 다름

B = 5000   # 부트스트랩 반복 횟수
for _ in range(B):
    # TODO: ____ tip_values에서 크기 n으로 복원추출
    resample = boot_rng.choice(tip_values, size=____, replace=____)
    # TODO: ____ 재표본의 평균을 저장
    boot_means.append(resample.____())

boot_means = np.array(boot_means)

# TODO: ____ 2.5%, 97.5% 백분위수로 95% 신뢰구간을 구하세요
ci_low, ci_high = np.percentile(boot_means, [____, ____])
print(f"팁 평균의 점추정치: {tip_values.mean():.3f}")
print(f"95% 부트스트랩 신뢰구간: [{ci_low:.3f}, {ci_high:.3f}]")

**[해석 질문 3]** 본인이 얻은 95% 신뢰구간을 한 문장으로 해석해 보세요("팁 평균이 ..."). 이때 **흔한 오해("참값이 이 구간에 있을 확률 95%")와 올바른 절차적 해석("이런 방식으로 구간을 반복해 만들면 그 중 약 95%가 참값을 포함")을 구분해서** 쓰세요. 그리고 옆 사람과 seed가 다르면 구간이 조금 다를 텐데, 그래도 되는 이유는 무엇일까요?

(여기에 답)

### 1.5 가설검정 — 두 집단 평균차 t검정 (유형 B)

개념: 두 집단(예: 점심 vs 저녁)의 평균 팁이 정말 다른지 판단합니다. **귀무가설(H0)**은 "두 집단 평균이 같다(차이=0)", **대립가설(H1)**은 "다르다"입니다. **t검정**은 관측된 평균차가 우연으로 보기엔 큰지를 재고, **p-value**로 결론냅니다. p-value는 "H0가 참일 때 이 정도 이상의 차이가 우연히 나올 확률"이며, 보통 0.05보다 작으면 H0를 기각(=차이 있다)합니다. 주의: **p-value는 데이터의 확률이지, H0가 참일 확률이나 H1이 참일 확률이 아닙니다** — 자주 하는 오해이니 구분하세요.

또 두 집단의 분산이 같다고 보기 어려울 때는 아래 코드처럼 `equal_var=False`(**Welch t검정**)를 쓰며, 이번에는 안전하게 이 옵션을 사용합니다. 이 `time` 비교는 p가 0.05 바로 근처의 **경계값**이라, 실행 환경/버전에 따라 출력이 기각/기각 실패 어느 쪽으로도 나올 수 있습니다 — 아래 코드의 `if`/`else` 두 분기는 모두 정상 코드이니, **본인 출력에 찍힌 결론을 그대로 보고**하세요.

In [ ]:
# time(Lunch/Dinner) 두 집단으로 tip 비교
group_dinner = tips.loc[tips["time"] == "Dinner", "tip"]
group_lunch  = tips.loc[tips["time"] == "Lunch",  "tip"]

print("Dinner 평균:", round(group_dinner.mean(), 3), "| n =", len(group_dinner))
print("Lunch  평균:", round(group_lunch.mean(), 3),  "| n =", len(group_lunch))

# TODO: ____ 독립표본 t검정을 수행하세요 (scipy stats.ttest_ind)
t_stat, p_val = stats.ttest_ind(group_dinner, group_lunch, equal_var=False)
print(f"\nt 통계량 = {t_stat:.3f}, p-value = {p_val:.4f}")

# TODO: ____ 유의수준 0.05 기준으로 결론을 print 하세요
alpha = 0.05
if p_val < ____:
    print("결론: 귀무가설 기각 -> 두 시간대의 평균 팁에 유의한 차이가 있다")
else:
    print("결론: 귀무가설 기각 실패 -> 유의한 차이가 있다고 말하기 어렵다")

**[해석 질문 4]** 코드가 출력한 결론(기각/기각 실패)은 무엇이었나요? 그 p-value는 0.05에서 얼마나 여유가 있었나요(경계에 가까웠나요)? 만약 `time` 대신 `smoker`(흡연 여부)로 tip을 비교하면 결과가 어떻게 될지 예상하고, 실제로 바꿔 돌려서 확인해 보세요. (셀을 추가해도 됩니다.)

(여기에 답)

### 1.6 단순선형회귀 (statsmodels OLS)

개념: **회귀(regression)**는 변수 관계를 수식으로 나타냅니다. 단순선형회귀는 `tip = 절편 + 기울기 × total_bill + 오차` 형태입니다. **OLS(최소제곱법)**는 실제 값과 예측 값의 차이(오차)의 제곱합을 최소로 만드는 절편·기울기를 찾는 방법입니다. `statsmodels`의 `.summary()`는 계수, p-value, 신뢰구간, R²를 한 번에 보여줍니다.

In [ ]:
# TODO: ____ tip을 total_bill로 설명하는 OLS 식을 작성하세요 (formula = "y ~ x")
model_simple = smf.ols(formula="____ ~ ____", data=tips).fit()

# TODO: ____ 요약표를 출력하세요
print(model_simple.____())

**[해석 질문 5]** `total_bill`의 계수(coef)는 얼마인가요? "청구액이 1달러 오르면 팁이 평균 ___달러 오른다"로 채워 해석하세요. 이 계수의 p-value와 95% 신뢰구간도 함께 언급하세요.

(여기에 답)

### 1.7 잔차 진단 (잔차 플롯 · Q-Q)

개념: **잔차(residual)**는 실제 값 − 모델 예측 값입니다. 좋은 선형회귀라면 잔차가 (1) 예측값과 무관하게 0 주변에 고르게 흩어지고 (2) 대략 정규분포를 따라야 합니다. **잔차 vs 예측값 플롯**으로 (1)을, **Q-Q 플롯**으로 (2)를 봅니다. Q-Q 플롯은 잔차가 정규분포라면 점들이 직선 위에 놓입니다.

In [ ]:
# TODO: ____ 적합값(예측값)과 잔차를 model_simple에서 꺼내세요
fitted = model_simple.____      # fittedvalues
resid  = model_simple.____      # resid

fig, axes = plt.subplots(1, 2, figsize=(11,4))

# (1) 잔차 vs 예측값
axes[0].scatter(fitted, resid, alpha=0.5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("예측값(fitted)")
axes[0].set_ylabel("잔차(resid)")
axes[0].set_title("잔차 플롯")

# (2) Q-Q 플롯
sm.qqplot(resid, line="45", fit=True, ax=axes[1])
axes[1].set_title("정규 Q-Q 플롯")
plt.show()

**[해석 질문 6]** 잔차 플롯에서 퍼짐이 예측값이 커질수록 넓어지는(깔때기 모양) 경향이 보이나요? **깔때기가 벌어지기 시작하는 대략적인 예측값 x좌표를 본인 그림에서 읽어** 적어 보세요. Q-Q 플롯의 점들은 직선에서 벗어나나요? 이 진단이 시사하는 문제(예: 등분산 가정 위반)를 한 가지 적어 보세요.

(여기에 답)

### 1.8 다중회귀 (범주형 one-hot 포함)

개념: 설명변수를 여러 개로 늘립니다. 범주형 변수(`sex`, `smoker`, `day`, `time`)는 숫자가 아니므로 **원-핫 인코딩(one-hot)**으로 0/1 더미 변수로 바꿔 넣습니다. statsmodels formula에서는 `C(변수)`가 이 처리를 자동으로 해 줍니다. 한 범주는 기준(baseline)으로 빠지고, 나머지 범주의 계수는 기준 대비 차이를 뜻합니다.

In [ ]:
# TODO: ____ total_bill, size + 범주형(sex, smoker, day, time)을 모두 넣은 다중회귀 식을 작성하세요
formula_multi = "tip ~ total_bill + size + C(sex) + C(smoker) + C(____) + C(____)"
model_multi = smf.ols(formula=formula_multi, data=tips).fit()
print(model_multi.summary())

### 1.9 계수 유의성 · R² / Adj R² · VIF

개념: **R²**는 모델이 `tip`의 변동을 몇 %나 설명하는지(0~1). **Adjusted R²**는 변수 개수를 벌점으로 반영한 값으로, 쓸모없는 변수를 넣으면 오히려 내려갑니다. **VIF(분산팽창인자)**는 설명변수끼리 서로 겹치는(다중공선성) 정도로, 보통 5(또는 10)를 넘으면 그 변수들이 서로 정보를 중복한다고 봅니다. VIF는 원래 모든 설명변수에 적용되지만, 여기서는 이해를 돕기 위해 수치형(`total_bill`, `size`)만 봅니다. 또 아래 표에 함께 찍히는 **상수항(const)의 VIF는 회귀 절편 계산 과정에서 생기는 수학적 산물이므로 해석 대상이 아니며 무시**합니다 — 실제로 볼 것은 `total_bill`과 `size` 두 행뿐입니다.

In [ ]:
# TODO: ____ 단순회귀 vs 다중회귀의 R^2, Adj R^2 비교
# 힌트: Adjusted R^2는 model.rsquared_adj 속성으로 얻는다
print(f"단순회귀 R^2 = {model_simple.rsquared:.4f}, Adj R^2 = {model_simple.____:.4f}")
print(f"다중회귀 R^2 = {model_multi.rsquared:.4f}, Adj R^2 = {model_multi.____:.4f}")

In [ ]:
# 수치형 설명변수들에 대한 VIF 계산
vif_df = tips[["total_bill", "size"]].copy()
vif_df = sm.add_constant(vif_df)   # 상수항 추가

vif_result = pd.DataFrame()
vif_result["변수"] = vif_df.columns
# TODO: ____ 각 열에 대해 variance_inflation_factor를 계산하세요
vif_result["VIF"] = [variance_inflation_factor(vif_df.values, i) for i in range(vif_df.shape[1])]
# 주의: 아래 표의 const(상수항) 행 VIF는 회귀 절편 때문에 생기는 수학적 산물이므로 무시하고,
#       total_bill과 size 행만 보고 해석하세요.
print(vif_result)

**[해석 질문 7]** 다중회귀에서 p-value가 0.05보다 작아 "유의한" 변수는 정확히 무엇무엇인가요? (`size`의 p-value가 0.05 바로 근처인지도 확인해 보세요.) `total_bill`과 `size`의 VIF는 걱정할 수준인가요? 단순→다중으로 갈 때 Adj R²는 올랐나요 내렸나요, 그 의미는?

(여기에 답)

### 1.10 해석 vs 예측 (train/test RMSE · MAE)

개념: 지금까지(statsmodels)는 "어떤 변수가 팁에 영향을 주나"를 **해석**했습니다. 이제 관점을 바꿔, 모델이 **본 적 없는 새 데이터를 얼마나 잘 맞히나(예측)**를 봅니다. 데이터를 train(학습)과 test(평가)로 나눠, train으로만 학습하고 test에서 오차를 잽니다. **RMSE**(예측오차 제곱평균의 제곱근)와 **MAE**(예측오차 절댓값 평균)가 대표 지표이며, 작을수록 좋습니다. 분할에는 본인 `MY_SEED`를 씁니다.

한 가지 짚을 점: 1.9에서 Adj R²로 보기엔 애매했던 범주형 변수라도, **해석용 변수 선택과 예측 성능은 별개의 관점**입니다. 그래서 여기서는 일부러 전체 변수를 넣어 예측 성능을 따로 확인해 봅니다(궁금하면 범주형을 빼고 RMSE가 어떻게 바뀌는지도 비교해 보세요).

In [ ]:
# 원-핫 인코딩된 설계행렬 만들기 (sklearn용)
# 주의: 목표변수 tip이 X에 새어들면 안 됨 -> 제거.
#   (심화 2.3을 먼저 돌렸다면 log_tip 열이 생겨 있을 수 있어 함께 방어적으로 제거한다.
#    정상 순서라면 아직 log_tip이 없으므로 errors="ignore" 덕에 그냥 무시된다.)
features = tips.drop(columns=["tip", "log_tip"], errors="ignore")
# TODO: ____ 범주형을 원-핫(get_dummies)으로 변환하세요 (drop_first=True로 기준범주 제거)
X = pd.get_dummies(features, drop_first=____).astype(float)
y = tips["tip"]

# TODO: ____ MY_SEED로 train/test 8:2 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=____
)
print("train:", X_train.shape, "| test:", X_test.shape)

In [ ]:
# TODO: ____ 선형회귀 모델을 train으로 학습하세요
lin = LinearRegression()
lin.fit(X_train, ____)

# TODO: ____ test에서 예측하세요
pred_test = lin.predict(____)

# TODO: ____ RMSE, MAE를 계산하세요
rmse = np.sqrt(mean_squared_error(y_test, pred_test))
mae  = mean_absolute_error(y_test, ____)
print(f"Test RMSE = {rmse:.3f}")
print(f"Test MAE  = {mae:.3f}")

**[해석 질문 8]** RMSE와 MAE 중 어느 쪽이 더 큰가요? 그 차이는 무엇을 의미할까요(힌트: RMSE는 큰 오차에 더 민감)? 그리고 statsmodels의 R²(해석)와 test RMSE(예측)는 서로 다른 질문에 답합니다. 두 방법의 목적 차이를 2~3문장으로 정리하세요.

(여기에 답)

### 1.11 결론

이번 프로젝트에서 얻은 결과를 스스로의 언어로 정리하세요. 아래 세 가지를 반드시 포함합니다.

1. 추론(신뢰구간 + t검정)으로 알게 된 것
2. 회귀(단순·다중)로 정량화한 관계와 유의한 변수
3. 해석용 모델과 예측용 모델의 결과가 서로 어떻게 연결/구분되는지

(여기에 결론 작성)

## 2. 심화 과제 (선택)

아래 중 **하나 이상**을 골라 수행하고, 결과를 해석 질문과 함께 정리하세요.

### 2.1 여러 seed로 신뢰구간 반복

서로 다른 seed 10개로 부트스트랩 95% 신뢰구간을 각각 구해, 구간이 얼마나 흔들리는지(그리고 대부분 서로 겹치는지) 관찰하세요.

In [ ]:
seeds = [MY_SEED + k for k in range(10)]
intervals = []
for s in seeds:
    r = np.random.default_rng(s)
    # TODO: ____ 위 1.4와 같은 방식으로 부트스트랩 평균 5000개를 만들고 CI를 구하세요
    bmeans = np.array([r.choice(tip_values, size=len(tip_values), replace=True).mean() for _ in range(____)])
    lo, hi = np.percentile(bmeans, [2.5, 97.5])
    intervals.append((s, lo, hi))

for s, lo, hi in intervals:
    print(f"seed={s}: [{lo:.3f}, {hi:.3f}]")

**[심화 질문]** seed마다 구간이 다르지만 폭과 위치는 비슷한가요? 이것이 신뢰구간을 신뢰해도 되는 이유와 어떻게 연결되나요?

(여기에 답)

### 2.2 변수 추가/제거로 R²·VIF 변화

다중회귀에서 변수를 하나씩 넣고 빼면서 Adj R²와 VIF가 어떻게 바뀌는지 실험하세요. (예: `size`를 뺀 모델, `total_bill`만 있는 모델 등)

In [ ]:
# TODO: ____ 최소 두 개 이상의 서로 다른 formula로 모델을 만들고 Adj R^2를 비교하세요
formulas = {
    "total_bill만": "tip ~ total_bill",
    "total_bill+size": "____",
    "full": formula_multi,
}
for name, f in formulas.items():
    m = smf.ols(f, data=tips).fit()
    print(f"{name:20s} | Adj R^2 = {m.rsquared_adj:.4f}")

### 2.3 로버스트 / 로그변환

1.7의 잔차 진단에서 깔때기 모양(등분산 위반)이 보였다면, 종속변수에 로그변환(`np.log`)을 적용하거나 로버스트 표준오차를 써 보고, 잔차 플롯이 **1.7의 원본 플롯과 비교해** 개선되는지 확인하세요. 추가로, `model_log.summary()`의 `total_bill` 계수를 **로그-선형 관점(청구액이 1달러 오르면 팁이 약 몇 % 변한다)**으로 한 문장 재해석해 보세요 — 로그변환은 잔차뿐 아니라 계수의 해석 단위(달러 → 퍼센트)도 바꿉니다.

In [ ]:
# TODO: ____ tip에 로그변환을 적용한 회귀를 적합하고 잔차 플롯을 다시 그려 비교하세요
tips["log_tip"] = np.log(tips["tip"])
model_log = smf.ols("log_tip ~ ____ + ____", data=tips).fit()

plt.figure(figsize=(5,4))
plt.scatter(model_log.fittedvalues, model_log.resid, alpha=0.5)
plt.axhline(0, color="red", linestyle="--")
plt.title("로그변환 후 잔차 플롯")
plt.show()

## 3. 회고

이번 과제에서 **가장 어려웠던 부분**과 그것을 어떻게 이해/해결했는지, 그리고 아직 헷갈리는 개념이 있다면 무엇인지 3~5문장으로 적으세요.

(여기에 회고 작성)

## 4. 생성형 AI 활용 방법

이 과제에서 GPT 등 생성형 AI를 어떻게 활용했는지 정리하고, **실제 대화 내역을 아래에 붙여넣으세요**.

- 어떤 부분에서 AI를 썼나요? (예: statsmodels formula 문법, VIF 코드, 오류 해결)
- AI가 준 답을 **본인이 어떻게 검증/판단**했나요? (그대로 믿지 않고 확인한 지점)
- AI가 틀리거나 애매했던 부분이 있었나요?

### 활용 방법 요약

(여기에 작성)

### 대화 내역 붙여넣기

```
(여기에 실제 프롬프트와 답변을 붙여넣기)
```